# Session 12 — Your turn: Read a paper, reproduce a claim

Two checks you can do on any paper's claim before running anything: is your reproduction compatible with it, and was the comparison fair?

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from aij_check import check, close, summary

## Task 1: Is my reproduction compatible with the claim?

A paper reports an improvement of **+2.0 points** (a standard error of 0.6 points, so a 95% interval of roughly +0.8 to +3.2). You reproduced it with five seeds and got these improvements: `mine = [0.4, 1.9, 1.1, 0.2, 0.9]`.
Compute `mine_mean`, its standard error `mine_se` (sample standard deviation divided by `sqrt(5)`), the interval `mine_interval` (mean plus or minus 2.776 standard errors, the t value for 4 degrees of freedom), and `compatible` (True if the two intervals overlap).

In [2]:
mine = np.array([0.4, 1.9, 1.1, 0.2, 0.9])
paper_interval = (0.8, 3.2)
mine_mean = mine.mean()
mine_se = mine.std(ddof=1) / np.sqrt(len(mine))
mine_interval = (mine_mean - 2.776 * mine_se, mine_mean + 2.776 * mine_se)
compatible = bool(mine_interval[0] <= paper_interval[1] and paper_interval[0] <= mine_interval[1])
print(mine_mean, mine_se, mine_interval, compatible)

0.9 0.2983286778035259 (np.float64(0.07183959041741217), np.float64(1.7281604095825878)) True


In [3]:
check("the mean improvement is 0.9", lambda: close(mine_mean, 0.9, 1e-9))
check("the standard error uses the sample standard deviation", lambda: close(mine_se, 0.2983, 0.001))
check("the interval is about 0.07 to 1.73", lambda: close(mine_interval, [0.072, 1.728], 0.01))
check("the intervals overlap: the reproduction does not contradict the paper, but it does not confirm +2.0 either", lambda: compatible is True)
summary()

PASS  the mean improvement is 0.9
PASS  the standard error uses the sample standard deviation
PASS  the interval is about 0.07 to 1.73
PASS  the intervals overlap: the reproduction does not contradict the paper, but it does not confirm +2.0 either

4 of 4 checks passed.  Well done.


**Why it works.** 'Reproduced' and 'not contradicted' are different statements. With five seeds the interval is wide; the honest summary is a smaller point estimate that the paper's interval still contains, and what you would need to learn more (more seeds, the paper's exact settings).

## Task 2: Was the comparison fair?

Compute the training cost `6 * N * D` of two setups: the baseline (`N = 124e6` parameters, `D = 300e6` tokens) and the paper's new method (`N = 124e6`, `D = 600e6` tokens). Store `cost_baseline`, `cost_new`, `ratio`.
Then decide which baseline budget makes the comparison fair: `fair_baseline_tokens` is the number of tokens the baseline needs for the **same** cost as the new method.

In [4]:
cost_baseline = 6 * 124e6 * 300e6
cost_new = 6 * 124e6 * 600e6
ratio = cost_new / cost_baseline
fair_baseline_tokens = cost_new / (6 * 124e6)
print(cost_baseline, cost_new, ratio, fair_baseline_tokens)

2.232e+17 4.464e+17 2.0 600000000.0


In [5]:
check("the baseline costs about 2.2e17 operations", lambda: close(cost_baseline / 2.2320e17, 1.0, 1e-3))
check("the new method costs twice as much", lambda: close(ratio, 2.0, 1e-9))
check("a fair baseline trains for 600 million tokens", lambda: close(fair_baseline_tokens, 600e6, 1.0))
summary()

PASS  the baseline costs about 2.2e17 operations
PASS  the new method costs twice as much
PASS  a fair baseline trains for 600 million tokens

3 of 3 checks passed.  Well done.


**Why it works.** A method that wins only because it was given twice the compute has shown nothing about the method. Session 12's paper map asks for equal budgets before any other question, because it is the cheapest flaw to spot and the commonest.